In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

SAVE_DIR = "/content/drive/MyDrive/ML_Assignment_Q3"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Results folder ready:", SAVE_DIR)

Results folder ready: /content/drive/MyDrive/ML_Assignment_Q3


In [4]:
from sklearn.datasets import fetch_olivetti_faces

data = fetch_olivetti_faces()

X = data.data
y = data.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of classes:", len(set(y)))

downloading Olivetti faces from https://ndownloader.figshare.com/files/5976027 to /root/scikit_learn_data
X shape: (400, 4096)
y shape: (400,)
Number of classes: 40


In [5]:
import numpy as np

rng = np.random.default_rng(26008)

train_indices = []
val_indices = []
test_indices = []

for identity in np.unique(y):
    identity_indices = np.where(y == identity)[0]
    identity_indices = rng.permutation(identity_indices)

    train_indices.extend(identity_indices[:6])
    val_indices.extend(identity_indices[6:8])
    test_indices.extend(identity_indices[8:10])

train_indices = np.array(train_indices)
val_indices = np.array(val_indices)
test_indices = np.array(test_indices)

X_train, y_train = X[train_indices], y[train_indices]
X_val, y_val = X[val_indices], y[val_indices]
X_test, y_test = X[test_indices], y[test_indices]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (240, 4096) (240,)
Validation: (80, 4096) (80,)
Test: (80, 4096) (80,)


In [6]:
X_test_masked = X_test.copy().reshape(-1, 64, 64)

X_test_masked[:, 24:40, 24:40] = 0

X_test_masked = X_test_masked.reshape(-1, 4096)

print("Masked test shape:", X_test_masked.shape)

Masked test shape: (80, 4096)


In [7]:
# Single Decision Tree
SINGLE_TREE_PARAMS = {
    "max_depth": None,
    "min_samples_leaf": 4
}

# Bagging
BAGGING_TREES = 200
BAGGING_TREE_PARAMS = {
    "max_depth": None,
    "min_samples_leaf": 4
}

# Boosting (SAMME)
BOOSTING_LEARNERS = 100
BOOSTING_MAX_DEPTH = 4

print("Single Tree:", SINGLE_TREE_PARAMS)
print("Bagging trees:", BAGGING_TREES)
print("Boosting learners:", BOOSTING_LEARNERS)
print("Boosting max depth:", BOOSTING_MAX_DEPTH)

Single Tree: {'max_depth': None, 'min_samples_leaf': 4}
Bagging trees: 200
Boosting learners: 100
Boosting max depth: 4


In [8]:
import numpy as np
import os

rng = np.random.default_rng(42)
resampled_datasets = []

for r in range(30):
    indices = []

    for identity in np.unique(y_train):
        identity_indices = np.where(y_train == identity)[0]

        sampled_indices = rng.choice(
            identity_indices,
            size=6,
            replace=True
        )

        indices.extend(sampled_indices)

    indices = np.array(indices)

    resampled_datasets.append(
        (X_train[indices], y_train[indices])
    )

print("Number of resampled datasets:", len(resampled_datasets))
print("First dataset shapes:", resampled_datasets[0][0].shape,
      resampled_datasets[0][1].shape)

Number of resampled datasets: 30
First dataset shapes: (240, 4096) (240,)


In [9]:
import numpy as np
import os

np.savez_compressed(
    f"{SAVE_DIR}/resampled_datasets.npz",
    X_resampled=np.array([item[0] for item in resampled_datasets]),
    y_resampled=np.array([item[1] for item in resampled_datasets])
)

print("30 resampled datasets saved to Drive!")

30 resampled datasets saved to Drive!


In [10]:
import numpy as np
import os
from sklearn.tree import DecisionTreeClassifier

single_path = f"{SAVE_DIR}/single_tree_results.npz"

# Pehle se saved results hain toh load kar lo
if os.path.exists(single_path):
    saved = np.load(single_path)
    single_tree_predictions = saved["predictions"]
    single_tree_accuracies = saved["accuracies"]
    print("Loaded saved Single Tree results.")

else:
    single_tree_predictions = []
    single_tree_accuracies = []

    for r, (X_resampled, y_resampled) in enumerate(resampled_datasets):
        model = DecisionTreeClassifier(
            max_depth=None,
            min_samples_leaf=4,
            random_state=42
        )

        model.fit(X_resampled, y_resampled)

        # Har model ki probabilities ko 40-class order mein align karo
        probabilities = np.zeros((len(X_test), 40))
        raw_probabilities = model.predict_proba(X_test)

        for j, cls in enumerate(model.classes_):
            probabilities[:, int(cls)] = raw_probabilities[:, j]

        predictions = np.argmax(probabilities, axis=1)
        accuracy = np.mean(predictions == y_test)

        single_tree_predictions.append(probabilities)
        single_tree_accuracies.append(accuracy)

        # Har completed refit ke baad save karo
        np.savez_compressed(
            single_path,
            predictions=np.array(single_tree_predictions),
            accuracies=np.array(single_tree_accuracies)
        )

        print(f"Refit {r+1}/30 complete | Accuracy: {accuracy:.4f}")

    single_tree_predictions = np.array(single_tree_predictions)
    single_tree_accuracies = np.array(single_tree_accuracies)

print("Prediction shape:", single_tree_predictions.shape)
print("Mean test accuracy:", single_tree_accuracies.mean())
print("Std. dev. test accuracy:", single_tree_accuracies.std())

Refit 1/30 complete | Accuracy: 0.5500
Refit 2/30 complete | Accuracy: 0.4625
Refit 3/30 complete | Accuracy: 0.4000
Refit 4/30 complete | Accuracy: 0.5375
Refit 5/30 complete | Accuracy: 0.4750
Refit 6/30 complete | Accuracy: 0.4500
Refit 7/30 complete | Accuracy: 0.4375
Refit 8/30 complete | Accuracy: 0.4750
Refit 9/30 complete | Accuracy: 0.5375
Refit 10/30 complete | Accuracy: 0.5125
Refit 11/30 complete | Accuracy: 0.4875
Refit 12/30 complete | Accuracy: 0.4125
Refit 13/30 complete | Accuracy: 0.5750
Refit 14/30 complete | Accuracy: 0.5125
Refit 15/30 complete | Accuracy: 0.5375
Refit 16/30 complete | Accuracy: 0.4625
Refit 17/30 complete | Accuracy: 0.5500
Refit 18/30 complete | Accuracy: 0.5250
Refit 19/30 complete | Accuracy: 0.4250
Refit 20/30 complete | Accuracy: 0.5000
Refit 21/30 complete | Accuracy: 0.4875
Refit 22/30 complete | Accuracy: 0.4375
Refit 23/30 complete | Accuracy: 0.5000
Refit 24/30 complete | Accuracy: 0.3875
Refit 25/30 complete | Accuracy: 0.4625
Refit 26/

In [11]:
mean_predictions = single_tree_predictions.mean(axis=0)

true_one_hot = np.eye(40)[y_test]

single_tree_bias_squared = np.mean(
    np.sum((mean_predictions - true_one_hot) ** 2, axis=1)
)

single_tree_variance = np.mean(
    np.sum(
        (single_tree_predictions - mean_predictions[None, :, :]) ** 2,
        axis=2
    )
)

print("Single Tree Bias²:", single_tree_bias_squared)
print("Single Tree Variance:", single_tree_variance)

Single Tree Bias²: 0.40480555555555553
Single Tree Variance: 0.6318611111111111


In [12]:
## bagging


In [13]:
print("resampled_datasets:", "resampled_datasets" in globals())
print("X_test:", "X_test" in globals())
print("y_test:", "y_test" in globals())
print("SAVE_DIR:", "SAVE_DIR" in globals())

resampled_datasets: True
X_test: True
y_test: True
SAVE_DIR: True


In [14]:
import numpy as np
import os
from sklearn.tree import DecisionTreeClassifier

bagging_path = f"{SAVE_DIR}/bagging_results.npz"

# Previously completed refits load karo, agar file exists karti hai
if os.path.exists(bagging_path):
    saved = np.load(bagging_path)
    bagging_predictions = list(saved["predictions"])
    bagging_accuracies = list(saved["accuracies"])
else:
    bagging_predictions = []
    bagging_accuracies = []

for r in range(len(bagging_predictions), 30):
    X_resampled, y_resampled = resampled_datasets[r]
    n = len(y_resampled)
    rng = np.random.default_rng(42)

    class_probabilities = np.zeros((200, len(X_test), 40))

    for b in range(200):
        indices = rng.choice(n, size=n, replace=True)

        tree = DecisionTreeClassifier(
            max_depth=None,
            min_samples_leaf=4,
            random_state=42 + b
        )
        tree.fit(X_resampled[indices], y_resampled[indices])

        probs = tree.predict_proba(X_test)
        for j, cls in enumerate(tree.classes_):
            class_probabilities[b, :, int(cls)] = probs[:, j]

    mean_probs = class_probabilities.mean(axis=0)
    predictions = np.argmax(mean_probs, axis=1)
    accuracy = np.mean(predictions == y_test)

    bagging_predictions.append(mean_probs)
    bagging_accuracies.append(accuracy)

    np.savez_compressed(
        bagging_path,
        predictions=np.array(bagging_predictions),
        accuracies=np.array(bagging_accuracies)
    )

    print(f"Refit {r+1}/30 complete | Accuracy: {accuracy:.4f}")

bagging_predictions = np.array(bagging_predictions)
bagging_accuracies = np.array(bagging_accuracies)

print("Prediction shape:", bagging_predictions.shape)
print("Mean test accuracy:", bagging_accuracies.mean())
print("Std. dev. test accuracy:", bagging_accuracies.std())

Refit 1/30 complete | Accuracy: 0.7625
Refit 2/30 complete | Accuracy: 0.8250
Refit 3/30 complete | Accuracy: 0.7625
Refit 4/30 complete | Accuracy: 0.8125
Refit 5/30 complete | Accuracy: 0.7750
Refit 6/30 complete | Accuracy: 0.7375
Refit 7/30 complete | Accuracy: 0.7875
Refit 8/30 complete | Accuracy: 0.7125
Refit 9/30 complete | Accuracy: 0.7875
Refit 10/30 complete | Accuracy: 0.7750
Refit 11/30 complete | Accuracy: 0.7500
Refit 12/30 complete | Accuracy: 0.7500
Refit 13/30 complete | Accuracy: 0.8000
Refit 14/30 complete | Accuracy: 0.7875
Refit 15/30 complete | Accuracy: 0.7250
Refit 16/30 complete | Accuracy: 0.7750
Refit 17/30 complete | Accuracy: 0.7375
Refit 18/30 complete | Accuracy: 0.7875
Refit 19/30 complete | Accuracy: 0.7875
Refit 20/30 complete | Accuracy: 0.7375
Refit 21/30 complete | Accuracy: 0.7250
Refit 22/30 complete | Accuracy: 0.7875
Refit 23/30 complete | Accuracy: 0.8000
Refit 24/30 complete | Accuracy: 0.8000
Refit 25/30 complete | Accuracy: 0.7625
Refit 26/

In [15]:
import numpy as np

mean_predictions = bagging_predictions.mean(axis=0)
true_one_hot = np.eye(40)[y_test]

bagging_bias_squared = np.mean(
    np.sum((mean_predictions - true_one_hot) ** 2, axis=1)
)

bagging_variance = np.mean(
    np.sum(
        (bagging_predictions - mean_predictions[None, :, :]) ** 2,
        axis=2
    )
)

print("Bagging Bias²:", bagging_bias_squared)
print("Bagging Variance:", bagging_variance)

Bagging Bias²: 0.5387637341641872
Bagging Variance: 0.018997354270130065


In [16]:
import os

print("SAVE_DIR exists:", os.path.exists(SAVE_DIR))
print("Resampled datasets available:", len(resampled_datasets))
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("Selected boosting settings:")
print("Learners:", BOOSTING_LEARNERS)
print("Max depth:", BOOSTING_MAX_DEPTH)

SAVE_DIR exists: True
Resampled datasets available: 30
X_test shape: (80, 4096)
y_test shape: (80,)
Selected boosting settings:
Learners: 100
Max depth: 4


In [17]:
import os

for file in os.listdir(SAVE_DIR):
    print(file)

resampled_datasets.npz
single_tree_results.npz
bagging_results.npz


In [18]:
import numpy as np
import os

for filename in [
    "resampled_datasets.npz",
    "single_tree_results.npz",
    "bagging_results.npz"
]:
    path = os.path.join(SAVE_DIR, filename)
    data = np.load(path)

    print(f"\n{filename}")
    for key in data.files:
        print(f"  {key}: shape = {data[key].shape}")


resampled_datasets.npz
  X_resampled: shape = (30, 240, 4096)
  y_resampled: shape = (30, 240)

single_tree_results.npz
  predictions: shape = (30, 80, 40)
  accuracies: shape = (30,)

bagging_results.npz
  predictions: shape = (30, 80, 40)
  accuracies: shape = (30,)


In [19]:
print("Boosting learners:", BOOSTING_LEARNERS)
print("Boosting max depth:", BOOSTING_MAX_DEPTH)
print("Selected model available:", "selected_boosting_model" in globals())
print("Resampled datasets:", len(resampled_datasets))

Boosting learners: 100
Boosting max depth: 4
Selected model available: False
Resampled datasets: 30


In [20]:
from sklearn.tree import DecisionTreeClassifier

print("Number of classes:", len(np.unique(y_train)))
print("Training datasets:", len(resampled_datasets))
print("Boosting learners per refit:", BOOSTING_LEARNERS)
print("Base tree max depth:", BOOSTING_MAX_DEPTH)


Number of classes: 40
Training datasets: 30
Boosting learners per refit: 100
Base tree max depth: 4


In [21]:

import numpy as np
from sklearn.tree import DecisionTreeClassifier

def train_samme(X, y, n_estimators=100, max_depth=4, random_state=42):
    classes = np.unique(y)
    K = len(classes)
    n = len(y)

    # Har training example ka initial weight equal hoga
    sample_weights = np.ones(n) / n

    models = []
    alphas = []

    rng = np.random.default_rng(random_state)

    for t in range(n_estimators):
        # Depth-4 decision tree as weak learner
        tree = DecisionTreeClassifier(
            max_depth=max_depth,
            random_state=random_state + t
        )

        tree.fit(X, y, sample_weight=sample_weights)
        predictions = tree.predict(X)

        # Weighted classification error
        error = np.sum(
            sample_weights * (predictions != y)
        )

        # SAMME: random multiclass prediction threshold
        if error >= 1 - (1 / K):
            # Agar first learner hi random se better nahi hai,
            # toh training fail hone ka signal return karo.
            if len(models) == 0:
                raise ValueError(
                    "First weak learner is not better than random."
                )
            break

        # Error ko numerical issues se safe rakho
        error = np.clip(error, 1e-12, 1 - 1e-12)

        # SAMME learner weight
        alpha = np.log((1 - error) / error) + np.log(K - 1)

        models.append(tree)
        alphas.append(alpha)

        # Sample weights update
        sample_weights *= np.exp(
            alpha * (predictions != y)
        )
        sample_weights /= sample_weights.sum()

        # Perfect classifier mil gaya toh stop
        if error == 0:
            break

    return {
        "models": models,
        "alphas": np.asarray(alphas),
        "classes": classes
    }

print("SAMME training function defined successfully.")


SAMME training function defined successfully.


In [22]:

def predict_samme(model, X):
    classes = model["classes"]
    models = model["models"]
    alphas = model["alphas"]

    n_samples = len(X)
    n_classes = len(classes)

    # S_k(x): har class ka weighted vote
    scores = np.zeros((n_samples, n_classes))

    for tree, alpha in zip(models, alphas):
        predictions = tree.predict(X)

        for j, cls in enumerate(classes):
            scores[:, j] += alpha * (predictions == cls)

    # q_k(x) = S_k(x) / sum(alpha_t)
    total_alpha = np.sum(alphas)

    if total_alpha > 0:
        confidence = scores / total_alpha
    else:
        confidence = np.zeros_like(scores)

    predicted_classes = classes[np.argmax(scores, axis=1)]

    return predicted_classes, confidence


In [23]:

import os
import numpy as np

boosting_path = f"{SAVE_DIR}/boosting_results.npz"

# Existing checkpoints load karo
if os.path.exists(boosting_path):
    saved = np.load(boosting_path)
    boosting_predictions = list(saved["predictions"])
    boosting_accuracies = list(saved["accuracies"])
    print(f"Resuming from refit {len(boosting_accuracies)}/30")
else:
    boosting_predictions = []
    boosting_accuracies = []

# Remaining refits complete karo
for r in range(len(boosting_accuracies), 30):
    X_resampled, y_resampled = resampled_datasets[r]

    model = train_samme(
        X_resampled,
        y_resampled,
        n_estimators=BOOSTING_LEARNERS,
        max_depth=BOOSTING_MAX_DEPTH,
        random_state=42 + r * 1000
    )

    predictions, confidence = predict_samme(model, X_test)
    accuracy = np.mean(predictions == y_test)

    boosting_predictions.append(confidence)
    boosting_accuracies.append(accuracy)

    # Har completed refit ke baad Drive mein save
    np.savez_compressed(
        boosting_path,
        predictions=np.array(boosting_predictions),
        accuracies=np.array(boosting_accuracies)
    )

    print(
        f"Refit {r+1}/30 complete | "
        f"Learners: {len(model['models'])} | "
        f"Accuracy: {accuracy:.4f}"
    )

boosting_predictions = np.array(boosting_predictions)
boosting_accuracies = np.array(boosting_accuracies)

print("\nPrediction shape:", boosting_predictions.shape)
print("Mean test accuracy:", boosting_accuracies.mean())
print("Std. dev. test accuracy:", boosting_accuracies.std())
print("Saved to:", boosting_path)


Refit 1/30 complete | Learners: 100 | Accuracy: 0.3875
Refit 2/30 complete | Learners: 100 | Accuracy: 0.3875
Refit 3/30 complete | Learners: 100 | Accuracy: 0.2375
Refit 4/30 complete | Learners: 100 | Accuracy: 0.3000
Refit 5/30 complete | Learners: 100 | Accuracy: 0.2125
Refit 6/30 complete | Learners: 100 | Accuracy: 0.2375
Refit 7/30 complete | Learners: 100 | Accuracy: 0.1750
Refit 8/30 complete | Learners: 100 | Accuracy: 0.4875
Refit 9/30 complete | Learners: 100 | Accuracy: 0.1750
Refit 10/30 complete | Learners: 100 | Accuracy: 0.4000
Refit 11/30 complete | Learners: 100 | Accuracy: 0.1750
Refit 12/30 complete | Learners: 100 | Accuracy: 0.3125
Refit 13/30 complete | Learners: 100 | Accuracy: 0.4250
Refit 14/30 complete | Learners: 100 | Accuracy: 0.3625
Refit 15/30 complete | Learners: 100 | Accuracy: 0.4125
Refit 16/30 complete | Learners: 100 | Accuracy: 0.2250
Refit 17/30 complete | Learners: 100 | Accuracy: 0.5125
Refit 18/30 complete | Learners: 100 | Accuracy: 0.2125
R

In [24]:
import numpy as np

mean_predictions = boosting_predictions.mean(axis=0)
true_one_hot = np.eye(40)[y_test]

boosting_bias_squared = np.mean(
    np.sum((mean_predictions - true_one_hot) ** 2, axis=1)
)

boosting_variance = np.mean(
    np.sum(
        (boosting_predictions - mean_predictions[None, :, :]) ** 2,
        axis=2
    )
)

print("Boosting Bias²:", boosting_bias_squared)
print("Boosting Variance:", boosting_variance)


Boosting Bias²: 0.8780863361954374
Boosting Variance: 0.017260994657569726


In [25]:

import os
import numpy as np
from sklearn.tree import DecisionTreeClassifier

data = np.load(f"{SAVE_DIR}/resampled_datasets.npz")
X_resampled_all = data["X_resampled"]
y_resampled_all = data["y_resampled"]

DEPTHS = {
    "shallow": 2,
    "selected": None,
    "deep": 16
}

single_depth_path = f"{SAVE_DIR}/single_tree_depth_comparison.npz"

print("Resampled X:", X_resampled_all.shape)
print("Resampled y:", y_resampled_all.shape)
print("Test set:", X_test.shape, y_test.shape)
print("Depth configurations:", DEPTHS)
print("Results will save to:", single_depth_path)


Resampled X: (30, 240, 4096)
Resampled y: (30, 240)
Test set: (80, 4096) (80,)
Depth configurations: {'shallow': 2, 'selected': None, 'deep': 16}
Results will save to: /content/drive/MyDrive/ML_Assignment_Q3/single_tree_depth_comparison.npz


In [26]:

import os
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# Har depth ke results save honge
if os.path.exists(single_depth_path):
    saved = np.load(single_depth_path)
    results = {
        key: {
            "predictions": list(saved[f"{key}_predictions"]),
            "accuracies": list(saved[f"{key}_accuracies"])
        }
        for key in DEPTHS
    }
    print("Existing checkpoints loaded.")
else:
    results = {
        key: {"predictions": [], "accuracies": []}
        for key in DEPTHS
    }

def save_single_depth_results():
    np.savez_compressed(
        single_depth_path,
        **{
            f"{key}_{metric}": np.array(results[key][metric])
            for key in DEPTHS
            for metric in ["predictions", "accuracies"]
        }
    )

# Har depth par 30 datasets
for name, depth in DEPTHS.items():
    start = len(results[name]["accuracies"])

    for r in range(start, 30):
        Xr = X_resampled_all[r]
        yr = y_resampled_all[r]

        model = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=4,
            random_state=42 + r
        )
        model.fit(Xr, yr)

        predictions = np.zeros((len(X_test), 40))
        probabilities = model.predict_proba(X_test)

        for j, cls in enumerate(model.classes_):
            predictions[:, int(cls)] = probabilities[:, j]

        accuracy = np.mean(
            np.argmax(predictions, axis=1) == y_test
        )

        results[name]["predictions"].append(predictions)
        results[name]["accuracies"].append(accuracy)

        save_single_depth_results()

        print(
            f"{name} | Refit {r+1}/30 | "
            f"Accuracy: {accuracy:.4f}"
        )

    print(f"Completed depth: {name}")

print("\nAll Single Tree depth experiments completed.")

for name in DEPTHS:
    acc = np.array(results[name]["accuracies"])
    print(
        f"{name}: mean accuracy = {acc.mean():.4f}, "
        f"std = {acc.std():.4f}"
    )


shallow | Refit 1/30 | Accuracy: 0.0750
shallow | Refit 2/30 | Accuracy: 0.0500
shallow | Refit 3/30 | Accuracy: 0.0500
shallow | Refit 4/30 | Accuracy: 0.0625
shallow | Refit 5/30 | Accuracy: 0.0750
shallow | Refit 6/30 | Accuracy: 0.0750
shallow | Refit 7/30 | Accuracy: 0.0750
shallow | Refit 8/30 | Accuracy: 0.0375
shallow | Refit 9/30 | Accuracy: 0.0750
shallow | Refit 10/30 | Accuracy: 0.0750
shallow | Refit 11/30 | Accuracy: 0.0500
shallow | Refit 12/30 | Accuracy: 0.0500
shallow | Refit 13/30 | Accuracy: 0.0750
shallow | Refit 14/30 | Accuracy: 0.0750
shallow | Refit 15/30 | Accuracy: 0.0500
shallow | Refit 16/30 | Accuracy: 0.0750
shallow | Refit 17/30 | Accuracy: 0.0625
shallow | Refit 18/30 | Accuracy: 0.0625
shallow | Refit 19/30 | Accuracy: 0.0500
shallow | Refit 20/30 | Accuracy: 0.0750
shallow | Refit 21/30 | Accuracy: 0.0625
shallow | Refit 22/30 | Accuracy: 0.0750
shallow | Refit 23/30 | Accuracy: 0.0750
shallow | Refit 24/30 | Accuracy: 0.0750
shallow | Refit 25/30 | A

In [27]:

true_one_hot = np.eye(40)[y_test]

for name in DEPTHS:
    predictions = np.array(results[name]["predictions"])

    mean_predictions = predictions.mean(axis=0)

    bias_squared = np.mean(
        np.sum(
            (mean_predictions - true_one_hot) ** 2,
            axis=1
        )
    )

    variance = np.mean(
        np.sum(
            (predictions - mean_predictions[None, :, :]) ** 2,
            axis=2
        )
    )

    print(f"\n{name.upper()}")
    print(f"Bias²: {bias_squared:.6f}")
    print(f"Variance: {variance:.6f}")



SHALLOW
Bias²: 0.903530
Variance: 0.047941

SELECTED
Bias²: 0.376500
Variance: 0.651833

DEEP
Bias²: 0.645458
Variance: 0.248947


In [28]:

import os
import numpy as np
from sklearn.tree import DecisionTreeClassifier

BAGGING_TREES_DEPTH = 100

bagging_depth_path = (
    f"{SAVE_DIR}/bagging_depth_comparison.npz"
)

print("Resampled datasets:", X_resampled_all.shape)
print("Test set:", X_test.shape, y_test.shape)
print("Depths:", DEPTHS)
print("Trees per ensemble:", BAGGING_TREES_DEPTH)
print("Checkpoint path:", bagging_depth_path)
print(
    "Existing checkpoint:",
    os.path.exists(bagging_depth_path)
)


Resampled datasets: (30, 240, 4096)
Test set: (80, 4096) (80,)
Depths: {'shallow': 2, 'selected': None, 'deep': 16}
Trees per ensemble: 100
Checkpoint path: /content/drive/MyDrive/ML_Assignment_Q3/bagging_depth_comparison.npz
Existing checkpoint: False


In [29]:

import os
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# Existing checkpoint load karo, agar available ho
if os.path.exists(bagging_depth_path):
    saved = np.load(bagging_depth_path)
    bagging_results = {
        name: {
            "predictions": list(saved[f"{name}_predictions"]),
            "accuracies": list(saved[f"{name}_accuracies"])
        }
        for name in DEPTHS
    }
    print("Existing results loaded.")
else:
    bagging_results = {
        name: {"predictions": [], "accuracies": []}
        for name in DEPTHS
    }

def save_bagging_depth_results():
    np.savez_compressed(
        bagging_depth_path,
        **{
            f"{name}_{metric}": np.array(bagging_results[name][metric])
            for name in DEPTHS
            for metric in ["predictions", "accuracies"]
        }
    )

for name, depth in DEPTHS.items():
    start = len(bagging_results[name]["accuracies"])

    for r in range(start, 30):
        Xr = X_resampled_all[r]
        yr = y_resampled_all[r]
        rng = np.random.default_rng(10000 + r)

        class_probabilities = np.zeros((100, 80, 40))

        for b in range(BAGGING_TREES_DEPTH):
            indices = rng.choice(len(yr), size=len(yr), replace=True)

            tree = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_leaf=4,
                random_state=10000 + r * 100 + b
            )
            tree.fit(Xr[indices], yr[indices])

            probs = tree.predict_proba(X_test)
            for j, cls in enumerate(tree.classes_):
                class_probabilities[b, :, int(cls)] = probs[:, j]

        mean_probs = class_probabilities.mean(axis=0)
        accuracy = np.mean(np.argmax(mean_probs, axis=1) == y_test)

        bagging_results[name]["predictions"].append(mean_probs)
        bagging_results[name]["accuracies"].append(accuracy)

        save_bagging_depth_results()

        print(
            f"{name} | Refit {r+1}/30 | "
            f"Accuracy: {accuracy:.4f}"
        )

    print(f"Completed depth: {name}")

print("\nAll Bagging depth experiments completed.")

for name in DEPTHS:
    acc = np.array(bagging_results[name]["accuracies"])
    print(
        f"{name}: mean accuracy = {acc.mean():.4f}, "
        f"std = {acc.std():.4f}"
    )


shallow | Refit 1/30 | Accuracy: 0.5125
shallow | Refit 2/30 | Accuracy: 0.4375
shallow | Refit 3/30 | Accuracy: 0.5375
shallow | Refit 4/30 | Accuracy: 0.4875
shallow | Refit 5/30 | Accuracy: 0.4500
shallow | Refit 6/30 | Accuracy: 0.5125
shallow | Refit 7/30 | Accuracy: 0.5250
shallow | Refit 8/30 | Accuracy: 0.4750
shallow | Refit 9/30 | Accuracy: 0.4750
shallow | Refit 10/30 | Accuracy: 0.5000
shallow | Refit 11/30 | Accuracy: 0.5000
shallow | Refit 12/30 | Accuracy: 0.5125
shallow | Refit 13/30 | Accuracy: 0.4750
shallow | Refit 14/30 | Accuracy: 0.5000
shallow | Refit 15/30 | Accuracy: 0.4625
shallow | Refit 16/30 | Accuracy: 0.4500
shallow | Refit 17/30 | Accuracy: 0.4875
shallow | Refit 18/30 | Accuracy: 0.4500
shallow | Refit 19/30 | Accuracy: 0.5250
shallow | Refit 20/30 | Accuracy: 0.5000
shallow | Refit 21/30 | Accuracy: 0.4250
shallow | Refit 22/30 | Accuracy: 0.5375
shallow | Refit 23/30 | Accuracy: 0.5375
shallow | Refit 24/30 | Accuracy: 0.4875
shallow | Refit 25/30 | A

In [30]:
##

In [11]:
import numpy as np
from sklearn.datasets import fetch_olivetti_faces

data = fetch_olivetti_faces()
y = data.target

# Recreate the same split using the original seed
rng = np.random.default_rng(26008)
train_indices, val_indices, test_indices = [], [], []

for identity in np.unique(y):
    identity_indices = np.where(y == identity)[0]
    identity_indices = rng.permutation(identity_indices)

    train_indices.extend(identity_indices[:6])
    val_indices.extend(identity_indices[6:8])
    test_indices.extend(identity_indices[8:10])

y_test = y[np.array(test_indices)]

print("y_test shape:", y_test.shape)

y_test shape: (80,)


In [12]:
import os

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))
print("Current directory:", os.getcwd())
print("Checkpoint folder exists:",
      os.path.exists("/content/drive/MyDrive/ML_Assignment_Q3"))

if os.path.exists("/content/drive/MyDrive/ML_Assignment_Q3"):
    print(os.listdir("/content/drive/MyDrive/ML_Assignment_Q3"))

Drive mounted: True
Current directory: /content
Checkpoint folder exists: True
['resampled_datasets.npz', 'single_tree_results.npz', 'bagging_results.npz', 'boosting_results.npz', 'single_tree_depth_comparison.npz', 'bagging_depth_comparison.npz']


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os

base_path = "/content/drive/MyDrive"

print("Drive mounted:", os.path.exists(base_path))

for root, dirs, files in os.walk(base_path):
    for file in files:
        if "bagging_depth_comparison" in file:
            print("Found:", os.path.join(root, file))

Drive mounted: True
Found: /content/drive/MyDrive/ML_Assignment_Q3/bagging_depth_comparison.npz


In [15]:
import numpy as np

bagging_depth_path = (
    "/content/drive/MyDrive/ML_Assignment_Q3/"
    "bagging_depth_comparison.npz"
)

saved = np.load(bagging_depth_path)

bagging_results = {}

for name in ["shallow", "selected", "deep"]:
    bagging_results[name] = {
        "predictions": list(saved[f"{name}_predictions"]),
        "accuracies": list(saved[f"{name}_accuracies"])
    }

    print(
        f"{name}: "
        f"{len(bagging_results[name]['accuracies'])} refits, "
        f"predictions shape = "
        f"{np.array(bagging_results[name]['predictions']).shape}"
    )

shallow: 30 refits, predictions shape = (30, 80, 40)
selected: 30 refits, predictions shape = (30, 80, 40)
deep: 30 refits, predictions shape = (30, 80, 40)


In [16]:
import numpy as np

true_one_hot = np.eye(40)[y_test]

for name in ["shallow", "selected", "deep"]:
    predictions = np.array(
        bagging_results[name]["predictions"]
    )

    mean_predictions = predictions.mean(axis=0)

    bias_squared = np.mean(
        np.sum(
            (mean_predictions - true_one_hot) ** 2,
            axis=1
        )
    )

    variance = np.mean(
        np.sum(
            (predictions - mean_predictions[None, :, :]) ** 2,
            axis=2
        )
    )

    accuracies = np.array(
        bagging_results[name]["accuracies"]
    )

    print(f"\n{name.upper()} BAGGING")
    print(f"Mean accuracy: {accuracies.mean():.4f}")
    print(f"Accuracy std:  {accuracies.std():.4f}")
    print(f"Bias squared:  {bias_squared:.6f}")
    print(f"Variance:      {variance:.6f}")


SHALLOW BAGGING
Mean accuracy: 0.4904
Accuracy std:  0.0338
Bias squared:  0.920021
Variance:      0.000995

SELECTED BAGGING
Mean accuracy: 0.7638
Accuracy std:  0.0402
Bias squared:  0.531387
Variance:      0.022641

DEEP BAGGING
Mean accuracy: 0.6971
Accuracy std:  0.0448
Bias squared:  0.666186
Variance:      0.012562


In [17]:
import numpy as np

# Load the saved Single Tree depth-comparison results
single_tree_path = (
    "/content/drive/MyDrive/ML_Assignment_Q3/"
    "single_tree_depth_comparison.npz"
)

single_saved = np.load(single_tree_path)

true_one_hot = np.eye(40)[y_test]

print(f"{'Depth':<12} {'Model':<12} {'Accuracy':>10} "
      f"{'Bias²':>12} {'Variance':>12}")

for name in ["shallow", "selected", "deep"]:
    # Single Tree predictions
    st_predictions = np.array(single_saved[f"{name}_predictions"])
    st_mean = st_predictions.mean(axis=0)

    st_bias = np.mean(
        np.sum((st_mean - true_one_hot) ** 2, axis=1)
    )
    st_variance = np.mean(
        np.sum((st_predictions - st_mean[None, :, :]) ** 2, axis=2)
    )
    st_accuracy = np.array(single_saved[f"{name}_accuracies"]).mean()

    # Bagging predictions
    bg_predictions = np.array(bagging_results[name]["predictions"])
    bg_mean = bg_predictions.mean(axis=0)

    bg_bias = np.mean(
        np.sum((bg_mean - true_one_hot) ** 2, axis=1)
    )
    bg_variance = np.mean(
        np.sum((bg_predictions - bg_mean[None, :, :]) ** 2, axis=2)
    )
    bg_accuracy = np.array(bagging_results[name]["accuracies"]).mean()

    print(f"{name:<12} {'Single Tree':<12} {st_accuracy:>10.4f} "
          f"{st_bias:>12.6f} {st_variance:>12.6f}")
    print(f"{'':<12} {'Bagging':<12} {bg_accuracy:>10.4f} "
          f"{bg_bias:>12.6f} {bg_variance:>12.6f}")

    reduction = st_variance - bg_variance
    reduction_pct = 100 * reduction / st_variance if st_variance else 0

    print(f"{'':<12} Variance reduction: {reduction:.6f} "
          f"({reduction_pct:.2f}%)\n")

Depth        Model          Accuracy        Bias²     Variance
shallow      Single Tree      0.0654     0.903530     0.047941
             Bagging          0.4904     0.920021     0.000995
             Variance reduction: 0.046946 (97.93%)

selected     Single Tree      0.4858     0.376500     0.651833
             Bagging          0.7638     0.531387     0.022641
             Variance reduction: 0.629193 (96.53%)

deep         Single Tree      0.2767     0.645458     0.248947
             Bagging          0.6971     0.666186     0.012562
             Variance reduction: 0.236386 (94.95%)



In [19]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

data = np.load(
    "/content/drive/MyDrive/ML_Assignment_Q3/resampled_datasets.npz"
)
X_resampled_all = data["X_resampled"]
y_resampled_all = data["y_resampled"]

depths = {
    "shallow": 2,
    "selected": None,
    "deep": 16
}

for name, depth in depths.items():
    leaf_sizes_all = []

    for r in range(30):
        tree = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=4,
            random_state=42 + r
        )
        tree.fit(X_resampled_all[r], y_resampled_all[r])

        is_leaf = tree.tree_.children_left == -1
        leaf_sizes = tree.tree_.n_node_samples[is_leaf]
        leaf_sizes_all.extend(leaf_sizes)

    leaf_sizes_all = np.asarray(leaf_sizes_all)

    print(f"\n{name.upper()}")
    print("Mean samples per leaf:",
          round(leaf_sizes_all.mean(), 3))
    print("Median samples per leaf:",
          round(np.median(leaf_sizes_all), 3))
    print("Fraction of leaves with exactly 4 samples:",
          round(np.mean(leaf_sizes_all == 4), 4))


SHALLOW
Mean samples per leaf: 78.261
Median samples per leaf: 6.0
Fraction of leaves with exactly 4 samples: 0.0

SELECTED
Mean samples per leaf: 6.0
Median samples per leaf: 6.0
Fraction of leaves with exactly 4 samples: 0.0

DEEP
Mean samples per leaf: 13.559
Median samples per leaf: 6.0
Fraction of leaves with exactly 4 samples: 0.0


In [20]:
print("Available variables:")
print([name for name in globals()
       if any(word in name.lower()
              for word in ["tree", "disagree", "prediction", "model"])])

Available variables:
['predictions', 'mean_predictions', 'single_tree_path', 'st_predictions', 'bg_predictions', 'DecisionTreeClassifier', 'tree']


In [21]:
for name in ["tree", "predictions", "mean_predictions",
             "st_predictions", "bg_predictions",
             "single_tree_path"]:
    obj = globals()[name]
    print(f"{name}: type={type(obj).__name__}, "
          f"shape={getattr(obj, 'shape', 'N/A')}")

tree: type=DecisionTreeClassifier, shape=N/A
predictions: type=ndarray, shape=(30, 80, 40)
mean_predictions: type=ndarray, shape=(80, 40)
st_predictions: type=ndarray, shape=(30, 80, 40)
bg_predictions: type=ndarray, shape=(30, 80, 40)
single_tree_path: type=str, shape=N/A


In [22]:
print(single_tree_path)


/content/drive/MyDrive/ML_Assignment_Q3/single_tree_depth_comparison.npz


In [23]:
import numpy as np

path = "/content/drive/MyDrive/ML_Assignment_Q3/bagging_depth_comparison.npz"

data = np.load(path)
print(data.files)

['shallow_predictions', 'shallow_accuracies', 'selected_predictions', 'selected_accuracies', 'deep_predictions', 'deep_accuracies']


In [24]:
import os

folder = "/content/drive/MyDrive/ML_Assignment_Q3"

for f in os.listdir(folder):
    print(f)

resampled_datasets.npz
single_tree_results.npz
bagging_results.npz
boosting_results.npz
single_tree_depth_comparison.npz
bagging_depth_comparison.npz


In [26]:
##d

In [27]:
import numpy as np
import os

folder = "/content/drive/MyDrive/ML_Assignment_Q3"

for filename in [
    "single_tree_results.npz",
    "bagging_results.npz",
    "boosting_results.npz"
]:
    path = os.path.join(folder, filename)
    data = np.load(path)

    print(f"\n--- {filename} ---")
    for key in data.files:
        arr = data[key]
        print(key, arr.shape, arr.dtype)


--- single_tree_results.npz ---
predictions (30, 80, 40) float64
accuracies (30,) float64

--- bagging_results.npz ---
predictions (30, 80, 40) float64
accuracies (30,) float64

--- boosting_results.npz ---
predictions (30, 80, 40) float64
accuracies (30,) float64


In [28]:
for name in [
    "best_model", "final_ensemble",
    "X_test", "y_test",
    "X_test_masked", "X_test_occluded",
    "X_test_mask", "X_masked_test"
]:
    if name in globals():
        obj = globals()[name]
        print(name, type(obj).__name__,
              getattr(obj, "shape", ""))

y_test ndarray (80,)


In [29]:
import os

for root, dirs, files in os.walk(
    "/content/drive/MyDrive/ML_Assignment_Q3"
):
    for f in files:
        print(os.path.join(root, f))

/content/drive/MyDrive/ML_Assignment_Q3/resampled_datasets.npz
/content/drive/MyDrive/ML_Assignment_Q3/single_tree_results.npz
/content/drive/MyDrive/ML_Assignment_Q3/bagging_results.npz
/content/drive/MyDrive/ML_Assignment_Q3/boosting_results.npz
/content/drive/MyDrive/ML_Assignment_Q3/single_tree_depth_comparison.npz
/content/drive/MyDrive/ML_Assignment_Q3/bagging_depth_comparison.npz


In [30]:
for name in ["best_model", "final_ensemble", "all_trees",
             "boosting_models", "models", "estimators"]:
    if name in globals():
        obj = globals()[name]
        print(name, type(obj).__name__)

In [31]:

import numpy as np
from sklearn.datasets import fetch_olivetti_faces
from sklearn.tree import DecisionTreeClassifier

# Dataset and original split
data = fetch_olivetti_faces()
X, y = data.data, data.target

rng = np.random.default_rng(26008)
train_idx, val_idx, test_idx = [], [], []

for identity in np.unique(y):
    idx = rng.permutation(np.where(y == identity)[0])
    train_idx.extend(idx[:6])
    val_idx.extend(idx[6:8])
    test_idx.extend(idx[8:10])

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

X_masked = X_test.reshape(-1, 64, 64).copy()
X_masked[:, 24:40, 24:40] = 0
X_masked = X_masked.reshape(-1, 4096)

# One selected single tree
single = DecisionTreeClassifier(
    max_depth=None, min_samples_leaf=4, random_state=42
)
single.fit(X_train, y_train)
single_clean = np.mean(single.predict(X_test) == y_test)
single_masked = np.mean(single.predict(X_masked) == y_test)

# One 200-tree bagging ensemble
rng_b = np.random.default_rng(42)
bag_probs = np.zeros((len(y_test), 40))

for b in range(200):
    idx = rng_b.choice(len(y_train), len(y_train), replace=True)
    tree = DecisionTreeClassifier(
        max_depth=None, min_samples_leaf=4, random_state=42 + b
    )
    tree.fit(X_train[idx], y_train[idx])
    probs = tree.predict_proba(X_test)
    bag_probs[:, tree.classes_.astype(int)] += probs / 200

bag_clean = np.mean(np.argmax(bag_probs, axis=1) == y_test)

bag_mask_probs = np.zeros((len(y_test), 40))
rng_b = np.random.default_rng(42)

for b in range(200):
    idx = rng_b.choice(len(y_train), len(y_train), replace=True)
    tree = DecisionTreeClassifier(
        max_depth=None, min_samples_leaf=4, random_state=42 + b
    )
    tree.fit(X_train[idx], y_train[idx])
    probs = tree.predict_proba(X_masked)
    bag_mask_probs[:, tree.classes_.astype(int)] += probs / 200

bag_masked = np.mean(np.argmax(bag_mask_probs, axis=1) == y_test)

# One SAMME boosting ensemble, using notebook's original functions
boost = train_samme(
    X_train, y_train, n_estimators=100,
    max_depth=4, random_state=42
)
boost_clean_pred, _ = predict_samme(boost, X_test)
boost_mask_pred, _ = predict_samme(boost, X_masked)

boost_clean = np.mean(boost_clean_pred == y_test)
boost_masked = np.mean(boost_mask_pred == y_test)

print("Single tree: clean", single_clean, "| masked", single_masked)
print("Bagging:     clean", bag_clean, "| masked", bag_masked)
print("Boosting:    clean", boost_clean, "| masked", boost_masked)


NameError: name 'train_samme' is not defined

In [32]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

def train_samme(X, y, n_estimators=100, max_depth=4,
                random_state=42):
    classes = np.unique(y)
    K = len(classes)
    n = len(y)
    weights = np.ones(n) / n
    models, alphas = [], []

    for t in range(n_estimators):
        model = DecisionTreeClassifier(
            max_depth=max_depth,
            random_state=random_state + t
        )
        model.fit(X, y, sample_weight=weights)
        pred = model.predict(X)
        error = np.sum(weights * (pred != y))

        if error >= 1 - 1 / K:
            break

        error = np.clip(error, 1e-12, 1 - 1e-12)
        alpha = np.log((1 - error) / error) + np.log(K - 1)
        models.append(model)
        alphas.append(alpha)

        weights *= np.exp(alpha * (pred != y))
        weights /= weights.sum()

        if error == 0:
            break

    return models, np.array(alphas), classes


def predict_samme(ensemble, X):
    models, alphas, classes = ensemble
    scores = np.zeros((len(X), len(classes)))

    for model, alpha in zip(models, alphas):
        pred = model.predict(X)
        for j, cls in enumerate(classes):
            scores[:, j] += alpha * (pred == cls)

    return classes[np.argmax(scores, axis=1)]


boost = train_samme(
    X_train, y_train,
    n_estimators=100,
    max_depth=4,
    random_state=42
)

boost_clean_pred = predict_samme(boost, X_test)
boost_mask_pred = predict_samme(boost, X_masked)

boost_clean = np.mean(boost_clean_pred == y_test)
boost_masked = np.mean(boost_mask_pred == y_test)

print(f"Boosting clean accuracy:  {boost_clean:.4f}")
print(f"Boosting masked accuracy: {boost_masked:.4f}")

Boosting clean accuracy:  0.5375
Boosting masked accuracy: 0.4125
